In [ ]:
# IMPORTS
import sys
import os
import torch
import torch.nn as nn
import scanpy as sc
import anndata as ad
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime as dt

import pandas, anndata, torch
print(pandas.__version__, anndata.__version__, torch.__version__)

# IMPORT OWN FILES
sys.path.append(os.path.abspath(".."))
import data_handling as dh
import binn_training as bt
from binn import BINN

# Supress warnings
import warnings
warnings.filterwarnings("ignore", message="Series.__getitem__ treating keys as positions", category=FutureWarning)
warnings.filterwarnings("ignore", message="The class AnnLoader is deprecated", category=FutureWarning)

In [ ]:
import importlib.util
from pathlib import Path

# Determine parent folder
base_dir = Path.cwd().parent
path = base_dir / "pipeline_paths_mnt.py"

# Import module dynamically
spec = importlib.util.spec_from_file_location("ppaths", path)
ppaths = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ppaths)

# to keep consistent paths
run_vars = "mg_200_mc_200_mhvg1000"
pp = ppaths.PipelinePaths("dev", full_pipeline_run_vars=run_vars)

In [ ]:
# GLOBALS
TRAIN_SIZE = 0.8
BATCH_SIZE = 32
MAX_EPOCHS = 20
FOLDS_OUTER = 3
FOLDS_INNER = 3
N_TRIALS = 30
EVAL_METRIC = 'roc_auc'

# PARAMETERS
LR = 0.002204
WEIGHT_DECAY = 0.021469
DROPOUT = 0.176345
ACTIVATION_FN = nn.Tanh()

# PATHS
data_path = pp.compl_full_pipe_path

# OTHER
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TUNE_HYPERPARAMETERS = True

now = dt.now().strftime("%y%m%d_%H%M")
if TUNE_HYPERPARAMETERS:
    now_hyper = now
else:
    # hyperparameters to read
    now_hyper = '261110_0958'

cell_types = ['Inh_combined']

all_results = {}
for cell_type in cell_types:
    print(f'RUNNING BINN PIPELINE FOR {cell_type}')
    data_info = f'{cell_type}_{run_vars}'
    HPARAM_PATH = pp.binn_model_path / f"{now_hyper}_best_hyperparameters_{data_info}.json"
    MASK_PATHS = [pp.mask_full_pipe_path / run_vars / f'{cell_type}_train_preprocessed_layer_{n}_mask.csv' 
              for n in range(5)]

    print("Reading masks...")
    masks = dh.read_masks(MASK_PATHS, print_shapes=True)

    print("Reading train and test data...")
    train_adata = ad.read_h5ad(data_path / f'{cell_type}_train_preprocessed.h5ad')
    test_adata = ad.read_h5ad(data_path / f'{cell_type}_test_preprocessed.h5ad')

    print(f'train_adata:\n{train_adata}')

    print("Roll-up to patient level...")
    patient_datasets = dh.rollup_to_patient_level(
        {'test':test_adata, 'train':train_adata}
        )

    print("Aligning adatas to BINN...")
    datasets_aligned = dh.subset_genes(patient_datasets, masks['df0'])

    print("Padding adatas to BINN-ready shape...")
    datasets_padded = dh.pad_align_data(datasets_aligned, masks["df0"])

    print("Getting dataloaders...")
    train_loader_global, test_loader_global = dh.create_dataloaders(
        datasets_padded['train'], 
        datasets_padded['test'], 
        batch_size=BATCH_SIZE
        )

    mask_matrix_list, in_features, layers_list, tensor_masks = dh.compute_features(masks, device)

    dh.poison_scanner(train_loader_global, device)
    dh.poison_scanner(test_loader_global, device)

    # CROSS VALIDATION
    outer_scores, outer_params = dh.run_CV_nested(
        datasets_padded['train'], 
        in_features, 
        layers_list, 
        tensor_masks,                                       
        device,
        eval_metric=EVAL_METRIC,
        k_outer=FOLDS_OUTER,
        k_inner=FOLDS_INNER,
        epochs=MAX_EPOCHS,
        n_trials=N_TRIALS,
        activation_fn=ACTIVATION_FN,
        save_path=pp.binn_model_path,
        data_info=data_info,
        time=now
        )

    # Hyperparameter tuning
    print("Performing hyperparameter tuning...")
    final = dh.hyperparameter_tuning_optuna(
        datasets_padded['train'], 
        in_features, 
        layers_list, 
        tensor_masks, 
        device, 
        batch_size=BATCH_SIZE, 
        activation_fn=ACTIVATION_FN,
        k=FOLDS_INNER, 
        epochs=MAX_EPOCHS,
        metric=EVAL_METRIC
        )
    print(f"Best parameters for BINN: {final}")
    dh.save_hyperparameters(final, HPARAM_PATH)

    # model is MLP with same layers as BINN 
    binn, criterion, optimizer, scheduler = dh.create_model(
        in_features, 
        layers_list, 
        tensor_masks, 
        device, 
        lr=final["lr"], 
        weight_decay=final["weight_decay"], 
        dropout=final["dropout"], 
        epochs=final['tune_total_epochs'],
        activation_fn=ACTIVATION_FN
        )
    print(binn)

    print(f"criterion: {criterion},\n optimizer: {optimizer}")

    # Run it
    dead_nodes_dict = dh.find_dead_outputs(binn, tensor_masks)
    print(dead_nodes_dict)

    # Dead output
    dh.show_dead_nodes(dead_nodes_dict, masks)

    for batch in train_loader_global:
        print(batch.X)
        break

    history = dh.train_fixed_epochs(
            binn, 
            train_loader_global, 
            criterion=criterion, 
            optimizer=optimizer, 
            scheduler=scheduler, 
            device=device,
            n_epochs=final['epochs'],
            save_path=pp.binn_model_path,
            data_info=data_info,
            time=now,
            monitor_loader=test_loader_global
            )

    # Harvest the results
    probs, targets, test_auc = dh.evaluate_model_roc(binn, test_loader_global, device)
    print(f"Test ROC-AUC: {test_auc:.4f}")

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    # Loss plot
    axes[0].plot(history['train_loss'], label='Training Loss', color='royalblue', linewidth=2)
    axes[0].plot(history['test_loss'], label='Testing Loss', color='orange', linestyle='--', linewidth=2)
    axes[0].set_title('Loss', fontsize=14)
    axes[0].set_xlabel('Epochs', fontsize=12)
    axes[0].set_ylabel('Loss (BCE)', fontsize=12)
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    # Accuracy plot
    axes[1].plot(history['train_acc'], label='Training Accuracy', color='green', linewidth=2)
    axes[1].plot(history['test_acc'], label='Testing Accuracy', color='red', linestyle='--', linewidth=2)
    axes[1].set_title('Accuracy', fontsize=14)
    axes[1].set_xlabel('Epochs', fontsize=12)
    axes[1].set_ylabel('Accuracy', fontsize=12)
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    plt.suptitle('BINN Training Diagnostics', fontsize=16)
    plt.tight_layout()
    fig.savefig(pp.binn_test_results_path / f'{now}_training_curves_{data_info}.png')
    plt.show()

    df_res = dh.save_test_results(
        binn, 
        test_loader_global, 
        device, 
        data_info, 
        pp.binn_test_results_path,
        now
        )
    
    dh.confusion_matrix_binn(df_res, pp.binn_test_results_path, data_info, now)

    report = dh.class_report(df_res)
    print(report)

    all_results[cell_type] = {
        'eval_metric': EVAL_METRIC,
        'nested_cv_mean': np.mean(outer_scores), 
        'nested_cv_std': np.std(outer_scores),
        'outer_epochs': [p['epochs'] for p in outer_params], 
        'final_epochs': final['epochs'],
    }
    
    pd.DataFrame(history).to_csv(
        pp.binn_model_path/ f'{now}_history_{data_info}.csv', 
        index_label='epoch')
    
    with open(pp.binn_test_results_path / f'{now}_class_report_{data_info}.txt', 'w') as f:
        f.write(report)

    print(f'COMPLETED BINN PIPELINE FOR {cell_type} \n\n\n')

summary = pd.DataFrame(all_results).T
summary.to_csv(pp.binn_test_results_path / f'{now}_summary_{run_vars}.csv')
summary

